In [ ]:
# uninstalled dependencies
%pip install datasketch hnswlib sentence-transformers

In [70]:
# Imports

import re
from itertools import combinations

import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH
import hnswlib
from sentence_transformers import SentenceTransformer
from collections import defaultdict

In [76]:
# Config

GOLD_FILE = 'PATSTAT_500_label_datav4.csv'
NAME_COL = 'comp'
GROUP_COL = 'Group'

EMBEDDING_MODEL = "Vsevolod/company-names-similarity-sentence-transformer" # embedding model purpose-built for company names
# MinHash parameters
MINHASH_THRESHOLD     = 0.2
MINHASH_PERM          = 128
SHINGLE_K             = 3

# HNSW parameters
HNSW_M                = 16
HNSW_K                = 15
HNSW_EF_CONSTRUCTION  = 100
HNSW_EF_SEARCH        = 100

# First-Token parameters
MAX_FIRST_TOKEN_BUCKET = 500

# sample size for the quick recall probe during tuning; None = full
TUNING_SAMPLE = None

In [51]:
# Name cleaning

LEGAL_FORMS = {
    "INC","INCORPORATED","CORP","CORPORATION","CO","COMPANY",
    "LTD","LIMITED","LLC","LP","PLC","GMBH","AG","KG","MBH",
    "SE","SA","SAS","SPA","SRL","BV","NV","AB","AS","ASA",
    "OY","OYJ","KK","KABUSHIKI","KAISHA","LTDA","PTY","THE",
    "AKTIENGESELLSCHAFT","GESELLSCHAFT",
}

ISO2 = {
    "US","GB","UK","DE","FR","JP","CN","KR","CA","AU","IT","NL","SE","CH","IL",
    "IN","BR","RU","ES","BE","AT","DK","FI","NO","TW","IE","SG","HK","NZ","ZA",
    "MX","AR","PL","CZ","HU","PT","GR","TR","MY","TH","PH","ID","VN","SA","AE",
    "EG","LU","IS","CY","EE","LV","LT","SI","SK","BG","RO","HR","RS","UA"
}

COUNTRY_NAMES = {
    "USA","UNITED STATES","UNITED STATES OF AMERICA","UK","UNITED KINGDOM",
    "GREAT BRITAIN","ENGLAND","SCOTLAND","WALES","GERMANY","DEUTSCHLAND","FRANCE"
    ,"JAPAN","NIPPON","NIHON","CHINA","PEOPLES REPUBLIC OF CHINA","PRC","KOREA",
    "SOUTH KOREA","REPUBLIC OF KOREA","TAIWAN","CANADA","AUSTRALIA","ITALY",
    "ITALIA","NETHERLANDS","HOLLAND","SWEDEN","SWITZERLAND","ISRAEL","INDIA",
    "BRAZIL","BRASIL","RUSSIA","RUSSIAN FEDERATION","SPAIN","BELGIUM","AUSTRIA",
    "DENMARK","FINLAND","NORWAY","IRELAND","SINGAPORE","HONG KONG","NEW ZEALAND",
    "MEXICO","ARGENTINA","POLAND","CZECH REPUBLIC","HUNGARY","PORTUGAL","GREECE",
    "TURKEY","MALAYSIA","THAILAND","PHILIPPINES","INDONESIA","VIETNAM","SAUDI ARABIA",
    "UNITED ARAB EMIRATES","LUXEMBOURG","ICELAND"
}

def _is_country(tok):
    return tok in ISO2 or tok in COUNTRY_NAMES

def strip_country(s):
    if not isinstance(s, str) or not s.strip():
        return ""
    s = re.sub(r"[^A-Z0-9 ,]", " ", s.upper())
    s = re.sub(r"\s+", " ", s).strip()
    if "," in s:
        head, tail = s.rsplit(",", 1)
        tt = tail.strip().split()
        if 1 <= len(tt) <= 4 and all(_is_country(t) for t in tt):
            s = head.strip()
    toks = s.split()
    if len(toks) >= 2:
        for n in (4, 3, 2, 1):
            if len(toks) >= n + 1:
                cand = " ".join(toks[-n:])
                if cand in COUNTRY_NAMES or (n == 1 and toks[-1] in ISO2):
                    toks = toks[:-n]
                    s = " ".join(toks)
                    break
    return re.sub(r"\s+", " ", s.replace(",", " ")).strip()

def clean_name(s):
    words = re.sub(r"[^A-Z0-9 ]", " ", strip_country(s).upper()).split()
    while words and words[-1] in LEGAL_FORMS:
        words.pop()
    while words and words[0] in LEGAL_FORMS:
        words.pop(0)
    return " ".join(words)

In [5]:
# load and clean gold standard file

df = pd.read_csv(GOLD_FILE)
df['clean'] = df[NAME_COL].apply(clean_name)
df = df[df['clean'].str.len() > 0].copy() # removes unrecoverable names

names = sorted(df['clean'].unique())
print(f"names: {len(names):,}  groups: {df[GROUP_COL].nunique():,}")

names: 2,968  groups: 481


In [73]:
# STAGE 1 (MinHash + SBERT and HNSW: from LSBlock): functions

def shingles(name, k=SHINGLE_K):
    s = name.lower()
    return {s[i:i+k] for i in range(len(s) - k + 1)} if len(s) >= k else {s}

def minhash_candidates(names):
    lsh = MinHashLSH(threshold=MINHASH_THRESHOLD, num_perm=MINHASH_PERM)
    sketches = {}
    for n in names:
        m = MinHash(num_perm=MINHASH_PERM)
        for sh in shingles(n):
            m.update(sh.encode("utf8"))
        sketches[n] = m
        lsh.insert(n, m)
    pairs = set()
    for n, m in sketches.items():
        for other in lsh.query(m):
            if other != n:
                pairs.add((n, other) if n < other else (other, n))
    return pairs

def semantic_candidates(names, model, k=HNSW_K):
    emb = model.encode(names, normalize_embeddings=True, show_progress_bar=True)
    index = hnswlib.Index(space="cosine", dim=emb.shape[1])
    index.init_index(max_elements=len(names),
                     ef_construction=HNSW_EF_CONSTRUCTION, M=HNSW_M)
    index.add_items(emb, np.arange(len(names)))
    index.set_ef(HNSW_EF_SEARCH)
    labels, _ = index.knn_query(emb, k=min(k + 1, len(names)))
    pairs = set()
    for i, neigh in enumerate(labels):
        for j in neigh:
            if j == i:
                continue
            a, b = names[i], names[int(j)]
            pairs.add((a, b) if a < b else (b, a))
    return pairs, emb

def blocking_recall(df, name_col, group_col, candidate_set):
    groups = df.groupby(group_col)[name_col].apply(lambda s: sorted(set(s.dropna())))
    total, covered, missed = 0, 0, []
    for members in groups:
        for a, b in combinations(members, 2):
            total += 1
            pair = (a, b) if a < b else (b, a)
            if pair in candidate_set:
                covered += 1
            else:
                missed.append(pair)
    return covered / total if total else 0.0, total, missed

def first_token_candidates(names, max_bucket=MAX_FIRST_TOKEN_BUCKET):
    buckets = defaultdict(list)
    for n in names:
        toks = n.split()
        if toks:
            buckets[toks[0]].append(n)
    pairs = set()
    for members in buckets.values():
        if 2 <= len(members) <= max_bucket:
            for a, b in combinations(members, 2):
                pairs.add((a, b) if a < b else (b, a))
    return pairs

In [33]:
# STAGE 1: MinHash

mh_pairs = minhash_candidates(names)
print(f"minhash pairs: {len(mh_pairs):,}")

minhash pairs: 146,733


In [63]:
# STAGE 1: SBERT + HNSW

#model = SentenceTransformer(EMBEDDING_MODEL)
model = SentenceTransformer()
hn_pairs, emb = semantic_candidates(names, model)
print(f"hnsw pairs: {len(hn_pairs):,}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/93 [00:00<?, ?it/s]

hnsw pairs: 34,024


In [74]:
# STAGE 1: Union and evaluation

ft_pairs = first_token_candidates(names)

union = mh_pairs | hn_pairs | ft_pairs
n_full = len(names) * (len(names)-1) / 2
print(f"first-token pairs: {len(ft_pairs):,}")
print(f"union pairs: {len(union):,} ({len(union)/n_full:.2%} of all pairs)")

recall, total, missed = blocking_recall(df, "clean", GROUP_COL, union)
print(f"blocking recall: {recall:.2%} ({total - len(missed):,}/{total:,})")

print(f"\nsample missed pairs ({len(missed)} total):")
for a, b in missed[:10]:
    print(f"  {a!r} vs {b!r}")

first-token pairs: 75,719
union pairs: 187,719 (4.26% of all pairs)
blocking recall: 97.56% (77,247/79,178)

sample missed pairs (1931 total):
  'NOKIA COPPORATION' vs 'SCANINTER NOKIA'
  'NOKIA DISPLAY PRODUCTS' vs 'SCANINTER NOKIA'
  'NOKIA HIGH SPEED ACCESS PRODUCTS' vs 'SCANINTER NOKIA'
  'NOKIA MOBILE DEVELOPMENTS LIMITED C O CLYDE' vs 'SCANINTER NOKIA'
  'NOKIA MOBILE PHONES' vs 'SCANINTER NOKIA'
  'NOKIA MOBILE PHONES U K' vs 'SCANINTER NOKIA'
  'NOKIA NETWOKS' vs 'SCANINTER NOKIA'
  'NOKIA SOLUTIONS AND NETWORKS MANAGEMENT' vs 'SCANINTER NOKIA'
  'NOKIA SOLUTIONS AND NETWORKS MANAGEMENT INTERNATIONAL' vs 'SCANINTER NOKIA'
  'NOKIA TECHNOLOGY' vs 'SCANINTER NOKIA'


In [79]:
# STAGE 1: bonus diagnosis

ft_recall, _, _ = blocking_recall(df,"clean",GROUP_COL, ft_pairs)
mh_recall, _, _ = blocking_recall(df, "clean", GROUP_COL, mh_pairs)
hn_recall, _, _ = blocking_recall(df, "clean", GROUP_COL, hn_pairs)
print(f"First-token alone recall: {ft_recall:.2%}")
print(f"MinHash alone recall: {mh_recall:.2%}")
print(f"HNSW alone recall:    {hn_recall:.2%}")
print(f"Union recall:         {blocking_recall(df, 'clean', GROUP_COL, union)[0]:.2%}")

First-token alone recall: 91.91%
MinHash alone recall: 57.12%
HNSW alone recall:    30.55%
Union recall:         97.56%


In [80]:
ft_only = ft_pairs - (mh_pairs | hn_pairs)
mh_only = mh_pairs - (ft_pairs | hn_pairs)
hn_only = hn_pairs - (ft_pairs | mh_pairs)

for label, s in [("first-token only", ft_only),
                 ("minhash only", mh_only),
                 ("hnsw only", hn_only)]:
    r, _, _ = blocking_recall(df, "clean", GROUP_COL, s)
    print(f"{label:20s}  pairs={len(s):>7,}  marginal recall={r:.2%}")

first-token only      pairs= 24,836  marginal recall=30.10%
minhash only          pairs=101,611  marginal recall=3.73%
hnsw only             pairs=  8,023  marginal recall=0.72%


In [82]:
# STAGE 2 (Label-free pair scoring) from DeepBlocker / UCL-Blocker: additional imports (TODO: if working merge with top cell)

import networkx as nx
import networkx.algorithms.community as nx_comm
from difflib import SequenceMatcher

In [83]:
# STAGE 2: config (if ok merge with top config cell)

SCORE_WEIGHTS = {"cosine": 0.55, "jaccard": 0.20, "edit": 0.20, "len_sim": 0.05}
SWEEP_PERCENTILES = (50, 60, 70, 75, 80, 85, 88, 90, 92, 95, 97, 98)
LOUVAIN_RESOLUTION = 1.0

In [84]:
# STAGE 2: Feature builder and scoring

def pair_features(pairs, emb_dict):
    rows = []
    for a, b in pairs:
        ea, eb = emb_dict[a], emb_dict[b]
        ta, tb = set(a.split()), set(b.split())
        la, lb = len(a), len(b)
        rows.append([
            float(np.dot(ea, eb)),
            len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0,
            SequenceMatcher(None, a, b).ratio(),
            1.0 - abs(la - lb) / max(la, lb, 1),
        ])
    df = pd.DataFrame(rows, columns=["cosine", "jaccard", "edit", "len_sim"])
    df["name_a"] = [a for a, b in pairs]
    df["name_b"] = [b for a, b in pairs]
    df["score"] = sum(df[k] * w for k, w in SCORE_WEIGHTS.items())
    return df

In [84]:
# STAGE 2: Feature builder and scoring

def pair_features(pairs, emb_dict):
    rows = []
    for a, b in pairs:
        ea, eb = emb_dict[a], emb_dict[b]
        ta, tb = set(a.split()), set(b.split())
        la, lb = len(a), len(b)
        rows.append([
            float(np.dot(ea, eb)),
            len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0,
            SequenceMatcher(None, a, b).ratio(),
            1.0 - abs(la - lb) / max(la, lb, 1),
        ])
    df = pd.DataFrame(rows, columns=["cosine", "jaccard", "edit", "len_sim"])
    df["name_a"] = [a for a, b in pairs]
    df["name_b"] = [b for a, b in pairs]
    df["score"] = sum(df[k] * w for k, w in SCORE_WEIGHTS.items())
    return df

In [88]:
# STAGE 2: scored candidate set

emb_dict = dict(zip(names, emb))
union_list = list(union)
scored = pair_features(union_list, emb_dict)
print(f"scored {len(scored):,} candidate pairs")
print(f"score distribution: min={scored['score'].min():.3f}  "
      f"median={scored['score'].median():.3f}  "
      f"p90={np.percentile(scored['score'], 90):.3f}  "
      f"max={scored['score'].max():.3f}")


# small correctness check
missing = [n for n in names if n not in emb_dict]
assert not missing, f"{len(missing)} names missing from emb_dict"
assert len(scored) == len(union_list), "scored must match union size"

scored 187,719 candidate pairs
score distribution: min=-0.117  median=0.251  p90=0.722  max=0.992


In [90]:
# STAGE 2: Modularity-based threshold sweep

def sweep_with_bcubed(scored, names, df, group_col, percentiles=SWEEP_PERCENTILES):
    y_true = (df.drop_duplicates("clean")
                .set_index("clean")[group_col]
                .reindex(names))
    rows = []
    for p in percentiles:
        thr = float(np.percentile(scored["score"], p))
        edges = scored[scored["score"] >= thr]
        G = nx.Graph()
        G.add_nodes_from(names)
        G.add_weighted_edges_from(
            (r.name_a, r.name_b, float(r.score)) for r in edges.itertuples()
        )
        comms = nx_comm.louvain_communities(
            G, weight="weight", resolution=LOUVAIN_RESOLUTION
        )
        cmap = {}
        for cid, c in enumerate(comms):
            for n in c:
                cmap[n] = cid
        y_pred = pd.Series({n: cmap[n] for n in names})
        p_, r_, f_ = bcubed(y_true, y_pred)
        sizes = sorted((len(c) for c in comms), reverse=True)
        rows.append({
            "pct": p,
            "threshold": round(thr, 3),
            "n_edges": len(edges),
            "n_comm": len(comms),
            "max_size": sizes[0],
            "singletons": sum(1 for s in sizes if s == 1),
            "bcubed_p": round(p_, 3),
            "bcubed_r": round(r_, 3),
            "bcubed_f": round(f_, 3),
        })
    return pd.DataFrame(rows)

bc_sweep = sweep_with_bcubed(scored, names, df, GROUP_COL)
print(bc_sweep.to_string(index=False))

 pct  threshold  n_edges  n_comm  max_size  singletons  bcubed_p  bcubed_r  bcubed_f
  50      0.251    93860      40       233           3     0.488     0.976     0.651
  60      0.419    75088     287       184         148     0.793     0.955     0.867
  70      0.604    56316     548       169         378     0.909     0.880     0.894
  75      0.652    46930     647       167         466     0.937     0.843     0.887
  80      0.680    37544     712       163         533     0.946     0.817     0.877
  85      0.701    28158     785       146         602     0.953     0.773     0.854
  88      0.713    22527     835       145         653     0.958     0.756     0.845
  90      0.722    18772     874       135         681     0.959     0.731     0.830
  92      0.731    15018     920       135         725     0.961     0.702     0.811
  95      0.748     9386    1074       129         855     0.965     0.615     0.751
  97      0.768     5632    1298        93        1024     0.978 

In [91]:
# STEP 2: Final clustering and B-Cubed

def cluster_map(comms):
    m = {}
    for cid, c in enumerate(comms):
        for n in c:
            m[n] = cid
    return m

def bcubed(y_true, y_pred):
    from collections import defaultdict
    truth, pred = defaultdict(set), defaultdict(set)
    for n in y_true.index:
        truth[y_true[n]].add(n)
        pred[y_pred[n]].add(n)
    P, R = [], []
    for n in y_true.index:
        tp, pp = truth[y_true[n]], pred[y_pred[n]]
        inter = len(tp & pp)
        P.append(inter / len(pp))
        R.append(inter / len(tp))
    p, r = float(np.mean(P)), float(np.mean(R))
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

best = sweep_df.loc[sweep_df["modularity"].idxmax()]
thr = float(best["threshold"])
edges = scored[scored["score"] >= thr]
G = nx.Graph()
G.add_nodes_from(names)
G.add_weighted_edges_from(
    (r.name_a, r.name_b, float(r.score)) for r in edges.itertuples()
)
comms = nx_comm.louvain_communities(G, weight="weight", resolution=LOUVAIN_RESOLUTION)
cmap = cluster_map(comms)

y_true = (df.drop_duplicates("clean")
            .set_index("clean")[GROUP_COL]
            .reindex(names))
y_pred = pd.Series({n: cmap[n] for n in names})
p, r, f = bcubed(y_true, y_pred)
print(f"chosen: pct={best['pct']}  threshold={thr:.3f}  "
      f"edges={best['n_edges']:,}  communities={best['n_comm']}")
print(f"BCubed  precision={p:.3f}  recall={r:.3f}  F1={f:.3f}")

chosen: pct=98.0  threshold=0.784  edges=3,755.0  communities=1534.0
BCubed  precision=0.983  recall=0.370  F1=0.538


In [ ]:
#REMEMBER TO CHANGE THRESHOLD!